In [1]:
import xarray as xr
import tifffile
import numpy as np
import glob, os, gc
import pandas as pd
import scanpy as sc

# Define functions

In [11]:
# write multi-channel tiff for CANVAS
def prep_canvas(datadir, outpath):
    print("Preparing data for CANVAS")
    
    files = glob.glob(f'{datadir}/10u/counts/*.nc')
    outdir = f'{outpath}/canvas/data/raw_data'
    os.makedirs(f'{outdir}', exist_ok=True)  
    os.makedirs(f'{outdir}/image_files', exist_ok=True)
    os.makedirs(f'{outdir}/../../configs/preprocess', exist_ok=True)

    for f in files:
        name = os.path.splitext(os.path.basename(f))[0].replace('.', '-')
        d = xr.open_dataarray(f)
        avgchannel = d.mean(dim="marker")
        avg_channel_expanded = avgchannel.expand_dims(dim={"marker": ["avg"]})
        d = xr.concat([d, avg_channel_expanded], dim="marker")

        # Save marker values to a text file
        with open(f"{outdir}/image_files/{name}.txt", "w") as txt_file:
            for marker in d.marker.values:
                txt_file.write(f"{marker}\n")

        # Save as a multi-channel TIFF
        arr = d.values  # shape (Y, X, channel)
        arr_tiff = np.transpose(arr, (2, 0, 1))  # shape (channel, Y, X)
        tifffile.imwrite(f"{outdir}/image_files/{name}.tif", arr_tiff)

    infile = glob.glob(f'{outdir}/image_files/*.txt')[0]
    outfile1 = f'{outdir}/common_channels.txt'
    outfile2 = f'{outdir}/../../configs/preprocess/channels_vis_strength.yaml'

    with open(infile, "r") as infile, open(outfile1, "w") as of1, open(outfile2, "w") as of2:
        for line in infile:
            of1.write(line)
            of2.write(line.strip() + ": 1\n")

In [5]:
# cells file for UTAG, CellCharter, and TissueMosaic
def prep_cells(cellspath, outpath, sid_name, cellid_name):
    print("Preparing Cells file")
    d = sc.read(cellspath)
    d = d[~d.obs.duplicated(subset=[sid_name, cellid_name], keep='first')]
    d.obs.rename(columns={sid_name: 'sid', cellid_name: 'cid'}, inplace=True)
    d.write(f'{outpath}/cells.h5ad')

In [21]:
# 100u spots file for STAGATE
def prep_spots100u(datadir, filename_parser, outpath):
    print("Preparing Spots file")
    
    # read in data and make into visium-sized spots
    files = glob.glob(f'{datadir}/10u/counts/*.nc')

    def downsample(data, sid, donor):
        y, x, c = data.shape
        new_y, new_x = y // 10, x // 10
        downsampled = data.data[:new_y * 10, :new_x * 10, :].reshape(new_y, 10, new_x, 10, c).sum(axis=(1, 3))
        y_coords = (np.arange(new_y) * 10 + 5) + data.coords['y'][0].data
        x_coords = (np.arange(new_x) * 10 + 5) + data.coords['x'][0].data
        
        downsampled = downsampled.reshape(-1, c)
        df = pd.DataFrame(downsampled, columns=data.coords['marker'].values)
        df['y'] = np.repeat(y_coords, new_x)
        df['x'] = np.tile(x_coords, new_y)
        df['sid'] = sid
        df['donor'] = donor
        return df

    samples = []
    for f in files:
        print('.', end='')
        s = xr.open_dataarray(f).astype(np.float32)
        s.attrs.update(filename_parser(f))
        samples.append(downsample(s, s.attrs['sid'], s.attrs['donor']))
    gc.collect()

    samples = pd.concat(samples)
    probes = samples.columns[:-4]
    genes = [g for g in probes if 'Blank' not in g]
    metadata = ['sid','donor', 'x','y']
    samples = samples[metadata + genes]

    # remove spots with few transcripts
    samples = samples[samples[genes].sum(axis=1) >= 500]
    samples.reset_index(inplace=True, drop=True)

    # create anndata object
    d = sc.AnnData(X=samples[genes],
                obs=samples[metadata],
                var=pd.DataFrame(index=genes))
    d.obsm['spatial'] = samples[['x','y']].values
    d.uns['spatial'] = samples[['x','y']].values

    # qc
    sc.pp.normalize_total(d, target_sum=np.median(d.X.sum(axis=1)))
    sc.pp.log1p(d)

    #save
    d.write(f'{outpath}/100u_spots.h5ad')

# Run

## ALZ

In [ ]:
def alz_filename_parser(fname):
    fname = os.path.splitext(os.path.basename(fname))[0]
    return {
        'donor': fname.split('_')[0],
        'sid': fname.split('_')[1]
    }
datadir = '../../ALZ/alz-data'
outpath = '_data/ALZ'

prep_cells(f'{datadir}/SEAAD_MTG_MERFISH.2024-12-11.h5ad', outpath,
           'Specimen Barcode', 'Cell ID')
prep_canvas(datadir, outpath)
prep_spots100u(datadir, alz_filename_parser, outpath)

Preparing Spots file
...........................................................................

/Users/yakir/miniconda3/envs/torch/lib/python3.12/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


# UC